# ☀️ Solar Photovoltaic (PV) Diagnostic EDA: Panel 1 vs. Panel 2

This notebook conducts a comprehensive **Exploratory Data Analysis (EDA)** separating **Panel 1 (String 1)** and **Panel 2 (String 2)** to analyze their electrical behavior, fault distributions, I-V / P-V curves, environmental correlations, and inter-panel differentials.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plotting aesthetics
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Load & Separate Panel 1 and Panel 2 Datasets

In [ ]:
# Load raw dataset with ';' delimiter
df_raw = pd.read_csv('PV_Diagnostic_Final_Labels.csv', sep=';')
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

# Panel 1 (String 1) DataFrame
df_p1 = df_raw[['timestamp', 'I1', 'V1', 'P1', 'T1', 'Irr', 'Label_Num_S1', 'Label_S1']].copy()
df_p1.columns = ['timestamp', 'Current', 'Voltage', 'Power', 'Temperature', 'Irradiance', 'Label_Num', 'Fault_Label']
df_p1['Panel'] = 'Panel 1'

# Panel 2 (String 2) DataFrame
df_p2 = df_raw[['timestamp', 'I2', 'V2', 'P2', 'T2', 'Irr', 'Label_Num_S2', 'Label_S2']].copy()
df_p2.columns = ['timestamp', 'Current', 'Voltage', 'Power', 'Temperature', 'Irradiance', 'Label_Num', 'Fault_Label']
df_p2['Panel'] = 'Panel 2'

print(f"Panel 1 shape: {df_p1.shape}")
print(f"Panel 2 shape: {df_p2.shape}")
display(df_p1.head())
display(df_p2.head())

## 2. Descriptive Statistics (Panel 1 vs Panel 2)

In [ ]:
print("=== Panel 1 Statistics ===")
display(df_p1[['Current', 'Voltage', 'Power', 'Temperature', 'Irradiance']].describe().T)

print("=== Panel 2 Statistics ===")
display(df_p2[['Current', 'Voltage', 'Power', 'Temperature', 'Irradiance']].describe().T)

## 3. Fault Class Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
p1_counts = df_p1['Fault_Label'].value_counts()
p2_counts = df_p2['Fault_Label'].value_counts()

axes[0].bar(p1_counts.index, p1_counts.values, color='#1f77b4', edgecolor='black', alpha=0.85)
axes[0].set_title("Panel 1: Fault Class Distribution", fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(p1_counts.values):
    axes[0].text(i, v + 1500, f"{v:,} ({v/len(df_p1)*100:.1f}%)", ha='center', fontsize=9)

axes[1].bar(p2_counts.index, p2_counts.values, color='#e06666', edgecolor='black', alpha=0.85)
axes[1].set_title("Panel 2: Fault Class Distribution", fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(p2_counts.values):
    axes[1].text(i, v + 1500, f"{v:,} ({v/len(df_p2)*100:.1f}%)", ha='center', fontsize=9)

plt.suptitle("Fault Class Distribution: Panel 1 vs Panel 2", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Mean Physical Parameters by Fault Class

In [ ]:
p1_grouped = df_p1.groupby('Fault_Label')[['Current', 'Voltage', 'Power', 'Temperature', 'Irradiance']].mean()
p2_grouped = df_p2.groupby('Fault_Label')[['Current', 'Voltage', 'Power', 'Temperature', 'Irradiance']].mean()

print("=== Panel 1: Mean Parameter Values by Fault ===")
display(p1_grouped)

print("=== Panel 2: Mean Parameter Values by Fault ===")
display(p2_grouped)

## 5. I-V (Current-Voltage) Characteristic Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
fault_labels = sorted(df_p1['Fault_Label'].unique())

for label in fault_labels:
    sub_p1 = df_p1[df_p1['Fault_Label'] == label].sample(min(2000, len(df_p1[df_p1['Fault_Label'] == label])), random_state=42)
    axes[0].scatter(sub_p1['Voltage'], sub_p1['Current'], label=label, alpha=0.4, s=12)

axes[0].set_title("Panel 1: I-V Curves by Fault Class", fontweight='bold')
axes[0].set_xlabel("Voltage (V)")
axes[0].set_ylabel("Current (A)")
axes[0].legend(loc='upper right')

for label in fault_labels:
    sub_p2 = df_p2[df_p2['Fault_Label'] == label].sample(min(2000, len(df_p2[df_p2['Fault_Label'] == label])), random_state=42)
    axes[1].scatter(sub_p2['Voltage'], sub_p2['Current'], label=label, alpha=0.4, s=12)

axes[1].set_title("Panel 2: I-V Curves by Fault Class", fontweight='bold')
axes[1].set_xlabel("Voltage (V)")
axes[1].legend(loc='upper right')

plt.suptitle("I-V Characteristic Comparison (Panel 1 vs Panel 2)", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. P-V (Power-Voltage) Characteristic Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

for label in fault_labels:
    sub_p1 = df_p1[df_p1['Fault_Label'] == label].sample(min(2000, len(df_p1[df_p1['Fault_Label'] == label])), random_state=42)
    axes[0].scatter(sub_p1['Voltage'], sub_p1['Power'], label=label, alpha=0.4, s=12)

axes[0].set_title("Panel 1: P-V Curves by Fault Class", fontweight='bold')
axes[0].set_xlabel("Voltage (V)")
axes[0].set_ylabel("Power (W)")
axes[0].legend(loc='upper left')

for label in fault_labels:
    sub_p2 = df_p2[df_p2['Fault_Label'] == label].sample(min(2000, len(df_p2[df_p2['Fault_Label'] == label])), random_state=42)
    axes[1].scatter(sub_p2['Voltage'], sub_p2['Power'], label=label, alpha=0.4, s=12)

axes[1].set_title("Panel 2: P-V Curves by Fault Class", fontweight='bold')
axes[1].set_xlabel("Voltage (V)")
axes[1].legend(loc='upper left')

plt.suptitle("P-V Characteristic Comparison (Panel 1 vs Panel 2)", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Correlation Analysis (Panel 1 vs Panel 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
corr1 = df_p1[['Current', 'Voltage', 'Power', 'Temperature', 'Irradiance']].corr()
corr2 = df_p2[['Current', 'Voltage', 'Power', 'Temperature', 'Irradiance']].corr()

im1 = axes[0].matshow(corr1, cmap='coolwarm', vmin=-1, vmax=1)
axes[0].set_title("Panel 1 Feature Correlations", fontweight='bold', pad=20)
axes[0].set_xticks(range(len(corr1.columns)))
axes[0].set_yticks(range(len(corr1.columns)))
axes[0].set_xticklabels(corr1.columns, rotation=45)
axes[0].set_yticklabels(corr1.columns)
for i in range(len(corr1.columns)):
    for j in range(len(corr1.columns)):
        axes[0].text(j, i, f"{corr1.iloc[i, j]:.2f}", ha="center", va="center", color="white" if abs(corr1.iloc[i, j]) > 0.5 else "black")

im2 = axes[1].matshow(corr2, cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_title("Panel 2 Feature Correlations", fontweight='bold', pad=20)
axes[1].set_xticks(range(len(corr2.columns)))
axes[1].set_yticks(range(len(corr2.columns)))
axes[1].set_xticklabels(corr2.columns, rotation=45)
axes[1].set_yticklabels(corr2.columns)
for i in range(len(corr2.columns)):
    for j in range(len(corr2.columns)):
        axes[1].text(j, i, f"{corr2.iloc[i, j]:.2f}", ha="center", va="center", color="white" if abs(corr2.iloc[i, j]) > 0.5 else "black")

fig.colorbar(im2, ax=axes, orientation='horizontal', fraction=0.05, pad=0.15)
plt.show()

## 8. Cross-Panel Fault Interdependence & Differential Diagnostics

In multi-panel PV systems, comparing Panel 1 with Panel 2 ($\\Delta I, \\Delta V, \\Delta P$) provides a powerful baseline diagnostic signal.

In [ ]:
print("=== Cross-Tabulation of Panel 1 vs Panel 2 Faults ===")
cross_tab = pd.crosstab(df_raw['Label_S1'], df_raw['Label_S2'], margins=True)
display(cross_tab)

# Calculate differentials
df_raw['Delta_I'] = df_raw['I1'] - df_raw['I2']
df_raw['Delta_V'] = df_raw['V1'] - df_raw['V2']
df_raw['Delta_P'] = df_raw['P1'] - df_raw['P2']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(df_raw['Delta_I'], bins=80, color='#3a86ff', edgecolor='black', alpha=0.7)
axes[0].set_title(r"$\\Delta I$ (I1 - I2)", fontweight='bold')
axes[0].set_xlabel("Current Diff (A)")

axes[1].hist(df_raw['Delta_V'], bins=80, color='#8338ec', edgecolor='black', alpha=0.7)
axes[1].set_title(r"$\\Delta V$ (V1 - V2)", fontweight='bold')
axes[1].set_xlabel("Voltage Diff (V)")

axes[2].hist(df_raw['Delta_P'], bins=80, color='#ff006e', edgecolor='black', alpha=0.7)
axes[2].set_title(r"$\\Delta P$ (P1 - P2)", fontweight='bold')
axes[2].set_xlabel("Power Diff (W)")

plt.suptitle("Inter-Panel Residuals / Differential Distributions", fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()